# 03 — Agents & Tools

**Goal**: Build an AI agent that can use tools to accomplish tasks.

Agents = LLM that decides which tools to call, when, and in what order.
This is the foundation of **AI assistants** (AutoGPT, Claude Computer Use, etc.).

In [ ]:
from langchain_community.llms import Ollama
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.prompts import PromptTemplate
import subprocess
import datetime

llm = Ollama(model="llama3.2", temperature=0.1)
print("Ready")

## 1. What is an Agent?

```
User: "What's the weather in Tokyo?"
                       │
                       v
    ┌───────────────────────────────────┐
    │           Agent (LLM)             │
    │  "I need to check the weather.    │
    │   I'll call the weather tool."    │
    └────────────┬──────────────────────┘
                 │
        ┌────────┴────────┐
        │   Tool Router   │
        └────────┬────────┘
                 │
     ┌───────────┼───────────┐
     v           v           v
  ┌──────┐  ┌────────┐  ┌────────┐
  │Search│  │Weather │  │Calc    │
  └──────┘  └────────┘  └────────┘
```

## 2. Creating Tools

A tool = a function with a name and description (the LLM reads the description to decide when to use it).

In [ ]:
def get_current_time(*args):
    """Get the current date and time."""
    return f"Current time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

def search_knowledge(query: str) -> str:
    """Search a knowledge base for information about GenAI."""
    knowledge = {
        "rag": "RAG = Retrieval Augmented Generation. Combines document retrieval with LLM generation.",
        "fine-tuning": "Fine-tuning adapts a pretrained model to a specific task using additional training.",
        "transformer": "Transformer architecture uses self-attention to process sequential data in parallel.",
        "embedding": "Embeddings are vector representations of text that capture semantic meaning.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"I don't have information about '{query}' in my knowledge base."

# Package as LangChain Tools
tools = [
    Tool(name="CurrentTime", func=get_current_time, description="Get the current date and time"),
    Tool(name="Calculator", func=calculator, description="Evaluate math expressions. Input: expression string"),
    Tool(name="KnowledgeBase", func=search_knowledge, description="Search for information about GenAI concepts like RAG, fine-tuning, transformers"),
]

print(f"Created {len(tools)} tools")

## 3. Zero-Shot ReAct Agent

The agent uses **ReAct** (Reasoning + Acting):
1. **Think**: "I need to figure out X"
2. **Act**: Call a tool
3. **Observe**: See the result
4. **Repeat** until done

In [ ]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=3
)

result = agent.run("What time is it right now?")
print(f"\nFinal answer: {result}")

In [ ]:
result = agent.run("Calculate 15 * 37 + 42")
print(f"\nFinal answer: {result}")

In [ ]:
result = agent.run("What is RAG and how is it different from fine-tuning?")
print(f"\nFinal answer: {result}")

## 4. Multi-Step Reasoning

Agents can chain multiple tool calls.

In [ ]:
result = agent.run(
    "What is the current time? Also calculate 2^10, " +
    "and then look up what an embedding is."
)
print(f"\nFinal answer: {result}")

## 5. When Agents Fail

Common failure modes:
1. **Agent loops**: keeps calling the same tool
2. **Wrong tool**: calls calculator for text questions
3. **Hallucinated tool**: tries to use a tool that doesn't exist
4. **Token limits**: too many iterations hits context window

Fixes:
- Set `max_iterations` (we used 3)
- Write better tool descriptions (they guide the LLM)
- Use `max_execution_time` to timeout

## 6. Custom Agent: Study Assistant

Let's build an agent that helps you study GenAI concepts.

In [ ]:
def create_flashcard(concept: str) -> str:
    """Create a flashcard for a concept."""
    prompt = f"""
    Create a flashcard for the concept '{concept}':
    Front: (question)
    Back: (answer, 1-2 sentences)
    """
    return llm.invoke(prompt)

def quiz_me(topic: str) -> str:
    """Generate a quiz question about a topic."""
    prompt = f"Generate a multiple-choice question about {topic}. Include 4 options and the correct answer."
    return llm.invoke(prompt)

study_tools = [
    Tool(name="CreateFlashcard", func=create_flashcard,
         description="Create a study flashcard for any concept"),
    Tool(name="QuizMe", func=quiz_me,
         description="Generate a quiz question about a topic to test your knowledge"),
    Tool(name="KnowledgeBase", func=search_knowledge,
         description="Look up information about GenAI concepts"),
]

study_agent = initialize_agent(
    tools=study_tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

result = study_agent.run("Help me study: first create a flashcard about RAG, then quiz me on it")
print(f"\nFinal: {result}")

## Key Takeaways

1. **Agents** = LLM that can call tools to accomplish tasks
2. **ReAct** = Reason → Act → Observe (the core agent loop)
3. **Tools** = functions with name + description (description is crucial)
4. **ZERO_SHOT_REACT_DESCRIPTION** = most common agent type
5. Agents are powerful but need guardrails (`max_iterations`, timeouts)

**Next**: Vector Databases — store and search embeddings efficiently →